## Collecting Account Followers

#### This script retrieves follower and activity information for accounts that appear in Bluesky starter packs. 

**Note:** In an earlier step of the pipeline, we collected (starter pack, account) pairs and stored them in a JSON file. Each entry represents an account that was included in a specific starter pack.

We collects:

- Follower count – used to measure the influence or audience size of the account.

- Post count – used as a rough indicator of account activity on the platform.

Although the post count is not used directly in the final analysis, it was collected to assess whether accounts included in starter packs are actively posting.

Therefore, these inforamtion will be store in another JSON file for futher analysis. Follower counts are later used to compute median follower values for each starter pack. These median values allow us to compare the typical influence level of accounts across different communities identified in the clustering analysis.

In [ ]:
# Import Libraries
import json
import pandas as pd
from atproto import Client, models
from atproto import exceptions
from password import BSKY_USERNAME, BSKY_APP_PASSWORD
import time

##### Loads the (Starter Pack, Account) pair JSONL:

In [ ]:
accounts = pd.read_json("./data/Trang_all_data.jsonl", lines=True)

In [8]:
accounts.head(10)

,SP URI,SP DID,SP Creator DID,SP Creator Handle,SP Description,Account DID,Account Handler
0,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:gbozpiwvs3rydrea7nvctqnk,cimadevilla69.bsky.social
1,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:e2vflw3j3tsy5ficuompjuuw,etienneklein.bsky.social
2,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:qqxqxgdu5z3he2piqfbfaku4,lemonde.fr
3,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:2egpzsea27fru2vkrjgdw2ob,mediapart.fr
4,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:ybx3s4azqenega7hdvxkioxa,edwyplenel.bsky.social
5,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:32yux6k4aht4wnvege7545pq,cecileduflot.bsky.social
6,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:kb7nqrzlrx3iap3p342iylrz,museedelhomme.fr
7,at://did:plc:hddh7iycnjbtlvcphysobl7n/app.bsky...,bafyreidfsz4rr3ezkiplxrngbtbeyqjsjrcteskeqoace...,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social,None,did:plc:hddh7iycnjbtlvcphysobl7n,fredleboucain.bsky.social
8,at://did:plc:3pduogkezfivisrbrj2qruki/app.bsky...,bafyreigiivq67qoxdfdo72riiaepwne45fxp3o57fb3h5...,did:plc:3pduogkezfivisrbrj2qruki,rivalmonarch.bsky.social,None,did:plc:mymwxdm4zedrqufkotuxn72k,tuyoki.bsky.social
9,at://did:plc:3pduogkezfivisrbrj2qruki/app.bsky...,bafyreigiivq67qoxdfdo72riiaepwne45fxp3o57fb3h5...,did:plc:3pduogkezfivisrbrj2qruki,rivalmonarch.bsky.social,None,did:plc:a3vzaavlv7pzsiygcp6bnfnw,piku.bsky.social


##### Extract all unique accounts:
There are duplicated accounts within the dataset and we only want to call BlueSky on unique accounts to save computation time.

In [4]:
# Extract unique accounts
unique_accounts = accounts["Account DID"].unique()

In [5]:
len(unique_accounts)

469260

In [ ]:
# Initial BlueSky Connection Session:
USERNAME = BSKY_USERNAME
APP_PASSWORD = BSKY_APP_PASSWORD

# Authenticate steps:
client = Client()
client.login(USERNAME, APP_PASSWORD)

ProfileViewDetailed(did='did:plc:lmc4xbbyqqyui7m6ptolv3lb', handle='tkieu137.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:lmc4xbbyqqyui7m6ptolv3lb/bafkreig5n2ooeo3ixz4yfygzcuablb4ovfm7fijbq7dorzbsfu2ezxrefy', banner=None, created_at='2026-01-13T22:59:12.525Z', debug=None, description=None, display_name='', followers_count=6, follows_count=75, indexed_at='2026-01-13T22:59:52.725Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=4, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=KnownFollower

##### Call getProfile API and Store info into a JSONL:

In [ ]:

############################################################
# Retrieve profile information for accounts appearing in starter packs
#
# This script iterates through a list of unique Bluesky account DIDs
# and queries the Bluesky API to collect profile statistics.
#
# The API endpoint `get_profiles` allows up to 25 actors per request,
# so the accounts are processed in batches of 25 to improve efficiency
# and avoid unnecessary API calls.
#
# For each account we collect:
#   - follower count (used to measure account influence)
#   - post count (used as a rough indicator of account activity)
#
# Results are appended to a JSONL file (`accounts_info.jsonl`) so that
# progress is saved continuously and the script can resume if interrupted.
#
# The script also handles API rate limits by detecting HTTP 429 errors
# and pausing execution before retrying the request.
# ##########################################################
accounts_info_list = []
start = 0
batch_size = 25

# batch = unique_accounts[start:start + batch_size]
# batch

while start < len(unique_accounts):
    batch = unique_accounts[start:start + batch_size].tolist()
    # print(batch)
    # start = start + 25
    try:
        accounts_info = client.app.bsky.actor.get_profiles({"actors": batch})

        with open("accounts_info.jsonl", "a") as f:
            for account in accounts_info.profiles:
                row = {
                    "Account DID": account.did,
                    "Account Followers Count": getattr(account, "followers_count", None),
                    "Account Posts Count": getattr(account, "posts_count", None)
                }

                accounts_info_list.append(row)
                f.write(json.dumps(row) + "\n")

        print(f"Processed rows {start} to {start + len(batch) - 1}")
        start += batch_size
        time.sleep(0.3)

    except Exception as e:
        print(f"Error at rows {start} to {start + len(batch) - 1}: {e}")

        if "429" in str(e) or "RateLimitExceeded" in str(e):
            print("Rate limit hit. Sleeping for 60 seconds...")
            time.sleep(60)
        else:
            print("Sleeping for 10 seconds, then retrying...")
            time.sleep(10)

Processed rows 0 to 24
Processed rows 25 to 49
Processed rows 50 to 74
Processed rows 75 to 99
Processed rows 100 to 124
Processed rows 125 to 149
Processed rows 150 to 174
Processed rows 175 to 199
Processed rows 200 to 224
Processed rows 225 to 249
Processed rows 250 to 274
Processed rows 275 to 299
Processed rows 300 to 324
Processed rows 325 to 349
Processed rows 350 to 374
Processed rows 375 to 399
Processed rows 400 to 424
Processed rows 425 to 449
Processed rows 450 to 474
Processed rows 475 to 499
Processed rows 500 to 524
Processed rows 525 to 549
Processed rows 550 to 574
Processed rows 575 to 599
Processed rows 600 to 624
Processed rows 625 to 649
Processed rows 650 to 674
Processed rows 675 to 699
Processed rows 700 to 724
Processed rows 725 to 749
Processed rows 750 to 774


KeyboardInterrupt: 